# 07 - Snapshots

**You will learn**: slowly changing dimensions (SCD type 2), `dbt snapshot`, the `timestamp` and `check` strategies.

**You will build**: the history of product prices and of sales person store assignments.

In bronze, `products` and `sales_persons` hold only the **current** state: when a price changes or a sales person moves to another store,
the row is **overwritten**. The history is lost, unless we keep it.

A **snapshot** records every version of a row (SCD type 2). Each time you run `dbt snapshot`, dbt compares the source with what it stored
and, for changed rows, closes the old version and inserts a new one:

| product_id | list_price | dbt_valid_from | dbt_valid_to |
|---|---|---|---|
| P0004 | 69.90 | 2023-06-01 | 2026-01-08 |
| P0004 | 73.40 | 2026-01-08 | *null* (= current) |

Snapshots need something to detect changes:

* **`timestamp` strategy**: the source has a reliable `updated_at` column (our case). Cheap and precise.
* **`check` strategy**: use it when there is no reliable `updated_at` (or you don't trust it). Instead of comparing a
  timestamp, dbt compares the current values of the columns you list (`check_cols: [list_price, unit_cost]`, or
  `check_cols: 'all'` to compare every column) against the last snapshotted version, and records a new version if any
  differ. It works without any special column in the source, but it is more expensive (every listed column, every run)
  and it can miss a change that reverts to a previous value **between two snapshot runs**, since there is no timestamp
  to order versions by - it only knows the row looks different now.

In [ ]:
from helpers import *

## 1. Define the snapshots

Snapshots can be defined **two ways**: as YAML under `snapshots/` (the current, recommended syntax), or as a `.sql` file with a
`{% snapshot %}` block (the older syntax, still fully supported). Both compile to the same thing - you'll write one of each so you
recognize both when you see them. `relation:` can point to a `source()` or a `ref()` - here we use a source, because we want
to capture the raw state **before** any dbt transformation, at the point where the source system last overwrote it.

**Exercise A.** Complete `snap_products` in `snapshots.yml` (YAML), then `snap_sales_persons` (unique key `sales_person_id`) in its
own `.sql` file using a `{% snapshot %}` block.

In [ ]:
%%writefile ../../src/snapshots/snapshots.yml
snapshots:
  - name: snap_products
    relation: source('bronze', 'products')
    config:
      database: silver
      tags: ['snapshot']
      # TODO: unique_key: the primary key of the product
      # TODO: strategy: timestamp
      # TODO: updated_at: the column that changes when the row changes


**Exercise A (continued).** Now `snap_sales_persons`, this time as a `.sql` file. The `config()` call inside the `{% snapshot %}`
block takes the same settings as the YAML `config:` block above - add, inside `config(...)`:

* `unique_key='sales_person_id'`
* `strategy='timestamp'`
* `updated_at='updated_at'`


In [ ]:
%%writefile ../../src/snapshots/snap_sales_persons.sql
{% snapshot snap_sales_persons %}

{{
    config(
        database='silver',
        tags=['snapshot'],
    )
}}

select * from {{ source('bronze', 'sales_persons') }}

{% endsnapshot %}


## 2. Take the baseline

The first `dbt snapshot` creates the tables with the current state of every row (one version each).
**Important:** history only starts at your first snapshot. Anything that changed before it is invisible.

In [ ]:
dbt("snapshot")

In [ ]:
batch_before = int(scalar(f"SELECT MAX(_batch_id) FROM bronze.sports_shop.sales_orders"))
base = q(f'''
    SELECT 'snap_products' AS snapshot, COUNT(*) AS versions, COUNT_IF(dbt_valid_to IS NULL) AS current_rows
    FROM silver.{SCHEMA}.snap_products
    UNION ALL
    SELECT 'snap_sales_persons', COUNT(*), COUNT_IF(dbt_valid_to IS NULL) FROM silver.{SCHEMA}.snap_sales_persons''')
base

One version per row, all current. dbt added four columns: `dbt_scd_id`, `dbt_updated_at`, `dbt_valid_from` and `dbt_valid_to`.
Run it again: with no change in the source, **nothing** should be added (snapshots are idempotent).

In [ ]:
dbt("snapshot")
q(f"SELECT COUNT(*) AS versions FROM silver.{SCHEMA}.snap_products")

## 3. Model the history

Two gold models expose the snapshots with friendlier names (`valid_from`, `valid_to`, `is_current`). They are provided.

In [ ]:
%%writefile ../../src/models/gold/dim_product_history.sql
-- Price history of every product, built on the snapshot (SCD type 2).
select
    product_id,
    product_name,
    category,
    brand,
    list_price,
    unit_cost,
    dbt_valid_from as valid_from,
    dbt_valid_to as valid_to,
    dbt_valid_to is null as is_current
from {{ ref('snap_products') }}


In [ ]:
%%writefile ../../src/models/gold/dim_sales_person_history.sql
-- Store assignments of every sales person over time, built on the snapshot (SCD type 2).
select
    sales_person_id,
    full_name,
    store_id,
    dbt_valid_from as valid_from,
    dbt_valid_to as valid_to,
    dbt_valid_to is null as is_current
from (
    select
        *,
        concat_ws(' ', first_name, last_name) as full_name
    from {{ ref('snap_sales_persons') }}
)


In [ ]:
%%writefile ../../src/models/gold/_gold.yml
version: 2
models:
- name: dim_date
  columns:
  - name: date_day
    data_tests:
    - unique
    - not_null
  - name: season
    data_tests:
    - accepted_values:
        arguments:
          values:
          - winter
          - spring
          - summer
          - autumn
- name: dim_product
  columns:
  - name: product_id
    data_tests:
    - unique
    - not_null
  - name: unit_margin
- name: dim_store
  columns:
  - name: store_id
    data_tests:
    - unique
    - not_null
- name: dim_sales_person
  columns:
  - name: sales_person_id
    data_tests:
    - unique
    - not_null
- name: dim_customer
  columns:
  - name: customer_id
    data_tests:
    - unique
    - not_null
  - name: country_name
    data_tests:
    - not_null
  - name: loyalty_tier
- name: dim_product_history
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - product_id
        - valid_from
  columns:
  - name: is_current
- name: dim_sales_person_history
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - sales_person_id
        - valid_from
  columns:
  - name: is_current
- name: fct_sales
  columns:
  - name: order_line_id
    data_tests:
    - unique
    - not_null
  - name: order_id
    data_tests:
    - not_null
  - name: order_day
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_date')
          field: date_day
  - name: customer_id
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_customer')
          field: customer_id
  - name: store_id
    data_tests:
    - not_null
    - relationships:
        arguments:
          to: ref('dim_store')
          field: store_id
  - name: sales_person_id
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_sales_person')
          field: sales_person_id
  - name: product_id
    data_tests:
    - not_null
    - relationships:
        arguments:
          to: ref('dim_product')
          field: product_id
  - name: order_status
    data_tests:
    - accepted_values:
        arguments:
          values:
          - completed
          - cancelled
          - returned
  - name: gross_amount
  - name: discount_amount
  - name: net_amount
    data_tests:
    - dbt_utils.accepted_range:
        arguments:
          min_value: 0
  - name: margin_amount
  - name: order_updated_at
- name: mart_revenue_by_category_month
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - month
        - category
  columns:
  - name: net_revenue
- name: mart_store_performance
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - month
        - store_id
- name: mart_sales_person_ranking
  columns:
  - name: sales_person_id
    data_tests:
    - unique
    - not_null
  - name: cumulative_revenue_share
- name: mart_revenue_by_country
  columns:
  - name: country_code
    data_tests:
    - unique
    - not_null


In [ ]:
dbt("build --select dim_product_history dim_sales_person_history")

## 4. The source changes

### Your trainer loads the next batch

The bronze data is **shared by the whole class**, so it is not loaded by you: your **trainer** runs the generator notebook once
(mode `append`, it changes about 5% of the product prices and moves about 5% of the sales persons to another store) and announces it. Nothing to do until then, except re-running the next cell every now and then.

**Important:** the trainer waits until everybody has taken the baseline snapshot above. Do not skip it, history starts there.

> **Going through this on your own, without a class?** Nobody will load the next batch for you - do it yourself: import
> `src/notebooks/generate_bronze_data.py` into your Databricks workspace and run it with `mode = append` (see TRAINER.md),
> either from the workspace UI, or from your own machine if you have it set up to run Databricks notebooks (e.g. the
> Databricks VS Code extension with Databricks Connect). Either way it still executes **on Databricks**, not in this
> notebook's local Python.

In [ ]:
# Re-run this cell until your trainer has announced the new batch
status = bronze_status()
check("a new batch is available in bronze", int(status.last_batch[0]) > batch_before,
      "not yet: wait for your trainer's announcement, then re-run this cell")
status

Capture the changes with `dbt snapshot`, then rebuild the two history models so that they read the new snapshot rows:

In [ ]:
dbt("snapshot")
dbt("build --select dim_product_history dim_sales_person_history")

In [ ]:
after = q(f'''
    SELECT 'snap_products' AS snapshot, COUNT(*) AS versions, COUNT_IF(dbt_valid_to IS NULL) AS current_rows
    FROM silver.{SCHEMA}.snap_products
    UNION ALL
    SELECT 'snap_sales_persons', COUNT(*), COUNT_IF(dbt_valid_to IS NULL) FROM silver.{SCHEMA}.snap_sales_persons''')
display(base)
display(after)
check("history was recorded", int(after.versions.sum()) > int(base.versions.sum()),
      "wait for the trainer's new batch (previous step), then run dbt('snapshot') again")

Which products changed price, and by how much?

In [ ]:
q(f'''
    SELECT product_id, product_name, list_price, valid_from, valid_to, is_current
    FROM (
        SELECT *, COUNT(*) OVER (PARTITION BY product_id) AS n FROM gold.{SCHEMA}.dim_product_history
    ) WHERE n > 1
    ORDER BY product_id, valid_from LIMIT 12
''')

And the sales persons who moved:

In [ ]:
q(f'''
    SELECT sales_person_id, full_name, store_id, valid_from, valid_to, is_current
    FROM (
        SELECT *, COUNT(*) OVER (PARTITION BY sales_person_id) AS n FROM gold.{SCHEMA}.dim_sales_person_history
    ) WHERE n > 1
    ORDER BY sales_person_id, valid_from
''')

**Exercise B. Point-in-time question.** Which store did each sales person work in on `2025-06-01`?
Complete the query (the version valid on that date has `valid_from <= date` and `valid_to` null or later than the date).

In [ ]:
q(f'''
    SELECT sales_person_id, full_name, store_id
    FROM gold.{SCHEMA}.dim_sales_person_history
    WHERE valid_from <= DATE'2025-06-01'
      -- TODO: and (valid_to is null or valid_to > DATE'2025-06-01')
    LIMIT 10
''')

## Discussion

* Why do snapshots read a **source** and not a silver model?
* What if the source had no `updated_at`? Use `strategy: check` with `check_cols: [list_price, unit_cost]`.
* What happens to a row **deleted** in the source? (Look up `hard_deletes` in the dbt docs.)
* How often should you run `dbt snapshot`? At least as often as the source can change, otherwise intermediate versions are lost.

## Recap

* Snapshots keep the history of rows that are overwritten in the source (SCD type 2).
* `timestamp` strategy needs a reliable `updated_at`; `check` compares columns.
* Ordering `dbt snapshot` and `dbt build` isn't a fixed rule - it depends what the snapshot feeds and how the source
  changes. Here, snapshotting before building means `dim_product_history`/`dim_sales_person_history` see the version
  captured in *this* run. Think it through for your own case (orchestration) rather than copying this order by default.

---

In [ ]:
# restore_checkpoint(7)